# Experimentos #104 — Parte 3: Entrada B (contexto + Top-3 chunks) sobre los datos de #104

A4 (#103, notebook 03) comparó la Entrada A (contexto + título/abstract del citado) con la B (contexto + `perfil : chunk` × Top-3
del citado) con solo **1.241 casos** de entrenamiento etiquetados por Haiku: SciBERT A→B = +0,005, IC95 [−0,021; +0,032]. Poca
data para descartar un efecto. Los notebooks 04/05 construyeron ~13k casos por pista (M: `natural` + `kd`; S: `sonnet`), pero
**solo corrieron la Entrada A**. Este notebook hace una comparación A vs B limpia sobre esos datos, en ambas pistas, y deja las
predicciones ciegas listas para el Test Gold.

**¿Se prueban los modelos de #104 con inputs B? No.** Fueron entrenados con A: darles chunks en la prueba mide cuánto falla un
modelo con un formato que nunca vio, no si los chunks ayudan. Para medir B hay que **entrenar** con B. Por eso se entrenan modelos
nuevos con A y con B sobre **exactamente las mismas citas, etiquetas, hiperparámetros y semillas** (el input es la única
diferencia). Los ganadores de #104 solo sirven de referencia: sus `predicciones_val.jsonl` (entrada A) restringidas a V_B, sin
reentrenar y sin input B.

> **Provisional**: no hay Test Gold humano. V_B sigue siendo fidelidad al maestro (QLoRA/Haiku/Sonnet); decide el Test Gold.

## Diseño fijado ANTES de ver cualquier resultado B

| Item | Valor |
|---|---|
| Pistas | M: `natural`, `modo="kd"` · S: `sonnet`, `modo="duro"` (los ganadores de #104) |
| Hiperparámetros | lr 3e-5, 4 épocas, paciencia 3, batch 8 (×2 acumulación), `max_len` 512 — **sin ajuste con B** |
| Semillas | 42, 1234, 2026 |
| Núcleo | 2 pistas × {A, B} × 3 semillas = 12 corridas de SciBERT |
| Opcional (según `HORAS_MAX`) | `B_barajada` × 2 pistas × 3 semillas; SPECTER A/B s42 × 2 pistas |
| Entrenamiento / validación | train ∩ cobertura B ∩ filtros de fuga; **V_B = V ∩ cobertura B**, usada para el early stopping en **A y en B** |
| MLflow | experimento `clasificacion-funcion-104-b` (no toca las tablas ni la reanudación del notebook 05) |

**Regla de decisión** (preregistrada). Primaria, por pista: diferencia B − A de F1-macro sobre V_B con etiqueta propia, por semilla y
promedio de semillas; bootstrap por clústeres (se remuestrean los artículos citantes). **«B ayuda en la pista X» solo si las 3 parejas
de semillas tienen el mismo signo _y_ el IC95 excluye 0.** Secundarias: val de la otra pista, `dev_haiku`, `ref_gpt` (∩ cobertura), F1
por clase, estratos citado visto/no visto en train, truncamiento, `B_barajada` vs B (efecto de la sección). Test Gold (después de
congelarlo): solo las parejas A vs B por pista y semilla, sobre Gold ∩ cobertura, ponderado y sin ponderar; sin elegir entre corridas.

**Salvedades.** Los resultados valen para citas cuyo citado es fragmentable (~55 %: la regla de A1 descarta artículos con párrafos de
más de 300 palabras) y la mezcla de clases cambia. La relevancia del Top-3 no está validada (#127): un resultado nulo puede reflejar la
calidad de la recuperación. Sobre Gold ∩ cobertura (~839 citas) el IC pareado será de ±0,04–0,06: un efecto pequeño no se resolverá allí.

Primero `DRY = True` (≤300 citas por conjunto, 64 casos y 1 época por run, experimento `-dry`); luego `DRY = False`.

In [1]:
import argparse
import dataclasses
import gc
import hashlib
import json
import statistics
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "packages/clasificacion-funcion/src"))
sys.path.insert(0, str(REPO / "packages/auditoria-corpus/src"))
sys.path.insert(0, str(REPO / "packages/recuperacion-densa/src"))
sys.path.insert(0, str(REPO))  # `src.tracking` y `proyecto_de_grado` (namespace desde la raiz)

import mlflow
import pandas as pd
import torch
from recuperacion_densa.cache_embeddings import CacheDisco

from clasificacion_funcion import adaptador_proyecto_grado as ad
from clasificacion_funcion import clasificador_torch as ct
from clasificacion_funcion import cli, cli_modelo, descarga_ocl, dominio
from clasificacion_funcion import entrada_b_104 as eb
from clasificacion_funcion import experimento_104 as ex
from clasificacion_funcion import registro_mlflow as rm
from clasificacion_funcion import seleccion_datos as sd
from clasificacion_funcion import servicio

/home/carlos/microproyecto-local-citation/local_workflows/.venv-qlora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DRY = False          # True: <=300 citas por conjunto, 64 casos y 1 epoca por run, experimento y carpetas aparte
DESCARGAR = True    # True SOLO de dia: baja de HuggingFace (ACL-OCL, publico; solo ids de articulos) los citados que faltan
HORAS_MAX = 10.0      # se omiten las fases restantes al superarlo
LIM_DRY = 300
L4_DESCARTAR = True      # L4: si hay chunks con solape, descarta esos citados del train/V (en A y B) y re-verifica; False = abortar
UMBRAL_L4 = eb.UMBRAL_SOLAPE   # 0.5 preregistrado (solo se baja para probar el camino de descarte en DRY)
CORRER_BARAJADA = True   # opcional: B con secciones barajadas (efecto de la seccion)
CORRER_SPECTER = True    # opcional: SPECTER A/B s42 (#103 exige el 2x2)

SEMILLAS_PLAN = (42, 1234, 2026)
SEMILLAS = (42, 1234) if DRY else SEMILLAS_PLAN
PISTAS = {"M": {"variante": "natural", "modo": "kd", "pista": "mixta"},
          "S": {"variante": "sonnet", "modo": "duro", "pista": "sonnet"}}
HP = dict(lr=3e-5, epocas=4, paciencia=3)   # config ganadora de #104; NO se ajusta con B
HP_OBJ = ct.Hiperparametros(**HP)
REFERENCIA_104 = {"M": "scibert-A-natural-kd-lr3e-05-e4", "S": "scibert-S-sonnet-lr3e-05-e4"}  # + "-s<semilla>"
N_BOOTSTRAP = 200 if DRY else 1000

SUFIJO = "-dry" if DRY else ""
ARTIFACTS = REPO / f"proyecto_de_grado/artifacts/experimentos_104_b{'_dry' if DRY else ''}"
EVIDENCIAS = REPO / f"proyecto_de_grado/docs/evidencias_experimentos_104_b{'_dry' if DRY else ''}"
CORRIDAS = ARTIFACTS / "corridas"
GOLD_DIR = ARTIFACTS / "gold_v2"
DIR_104 = REPO / "proyecto_de_grado/artifacts/experimentos_104/corridas"
EVIDENCIAS.mkdir(parents=True, exist_ok=True)
T0 = time.time()
FALLAS = []
LISTA_RUNS = []   # {nombre, encoder, entrada, pista, semilla, run_id} de cada run lanzado

# En DRY los artefactos de MLflow van a una carpeta *_dry (borrable), no a `mlartifacts/` de la raiz
rm.configurar(rm.EXPERIMENTO_B + SUFIJO, artifacts_dir=(ARTIFACTS / "mlartifacts") if DRY else None)
CLIENTE = mlflow.MlflowClient()
print("tracking:", mlflow.get_tracking_uri(), "| DRY =", DRY, "| DESCARGAR =", DESCARGAR)


def hay_tiempo() -> bool:
    return (time.time() - T0) / 3600 < HORAS_MAX


def liberar_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def escribir_json(nombre: str, obj) -> Path:
    ruta = EVIDENCIAS / nombre
    ruta.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return ruta


def correr(nombre, fn, *a, **kw):
    # Ejecuta un run sin detener el notebook: la falla queda FAILED en MLflow y en FALLAS.
    # Ante `OutOfMemoryError` reintenta una vez con micro-lote 4 y acumulacion 4 (mismo lote efectivo).
    if not hay_tiempo():
        print(f"[omitido por HORAS_MAX] {nombre}")
        return None
    t = time.time()
    for intento in (1, 2):
        try:
            run_id = fn(nombre, *a, **kw)
            print(f"[ok {time.time() - t:6.0f}s] {nombre} -> {run_id}")
            return run_id
        except torch.OutOfMemoryError:
            print(f"[OOM] {nombre} (intento {intento})")
            if intento == 1 and "hp" in kw:
                kw = {**kw, "hp": dataclasses.replace(kw["hp"], batch_size=4, acumulacion=4)}
                continue
            FALLAS.append(nombre)
            return None
        except Exception:
            FALLAS.append(nombre)
            print(f"[FALLO] {nombre}\n{traceback.format_exc()}")
            return None
        finally:
            liberar_gpu()

tracking: sqlite:////home/carlos/microproyecto-local-citation/mlflow.db | DRY = False | DESCARGAR = True


## Fase 0 — Datos de #104, controles L1–L3 y preregistro

- **L1** reglas de contexto de #104 (ids, citante, par, texto, años 2014/2015, Gold v2 + TG2700 + taller/piloto/calibración): `sd.construir_todo` falla cerrado.
- **Fuga propia de la Entrada B** (hallada al planificar): 388 citas `natural` y 392 `sonnet` citan un artículo que es *citante de V*. Con B su input
  llevaría chunks del texto completo de ese artículo, es decir, frases de citas de V dentro de entradas de entrenamiento. **L2**: se descartan de
  entrenamiento (en A **y** en B, para que ambas usen las mismas citas) las citas cuyo citado es citante de V ∪ Gold v2 ∪ TG2700 ∪ excluidos.
- **L3** ninguna cita de V cita a un citante del Gold.
- **L7** el Test Gold anotado (`TEST_GOLD_2700_ANOTADO.csv`) **no se carga nunca**: del Gold solo se leen ids (manifiestos).

In [3]:
FUENTES = ad.cargar_fuentes_104()
PAPERS = ad.cargar_papers()
CTX = FUENTES.contextos
GOLD_V2_IDS = sorted(ad.FuentesProyectoGrado().ids_gold_v2())  # solo ids del manifiesto (L7)
assert FUENTES.sonnet, "Faltan las etiquetas Sonnet consolidadas: corra la fase 0 del notebook 05"

# L1 (lanza ValueError si hay cualquier fuga)
RES = sd.construir_todo(FUENTES)
assert RES["fugas"]["total_fugas"] == 0 and RES["fugas"]["citas_de_anios_excluidos"] == 0
V, V_SONNET, VAR = RES["v"], RES["v_sonnet"], RES["variantes"]

# Los MISMOS datos que usaron los notebooks 04/05 (si cambian, este analisis ya no seria sobre #104)
M104 = json.loads((REPO / "proyecto_de_grado/docs/evidencias_experimentos_104/manifiesto_datos.json").read_text(encoding="utf-8"))
SHA_ACTUAL = {n: dominio.sha_ids(x["context_id"] for x in filas)
              for n, filas in {"natural": VAR["natural"], "sonnet": VAR["sonnet"], "validacion": V,
                               "validacion_sonnet": V_SONNET}.items()}
assert SHA_ACTUAL == {n: M104["sha256_ids"][n] for n in SHA_ACTUAL}, "Los datos ya no coinciden con #104"

# L2 / L3
CITANTES = eb.citantes_evaluacion(FUENTES, V)
TRAIN_M, QUITADAS_M = eb.excluir_citas_a_citantes(VAR["natural"], CTX, CITANTES)
TRAIN_S, QUITADAS_S = eb.excluir_citas_a_citantes(VAR["sonnet"], CTX, CITANTES)
L3 = eb.verificar_citados_v(V, CTX, eb.citantes_gold(FUENTES))
L1 = sd.verificar({"natural": TRAIN_M, "sonnet": TRAIN_S}, V, FUENTES)   # se vuelve a verificar tras L2
assert L1["total_fugas"] == 0
print(f"L2 quita {sum(QUITADAS_M.values())} de {len(VAR['natural'])} (natural) y {sum(QUITADAS_S.values())} de {len(VAR['sonnet'])} (sonnet)")
print("  natural por clase:", QUITADAS_M)
print("  sonnet  por clase:", QUITADAS_S)
print("L3:", L3)

# Subconjuntos de la prueba en seco (los datos completos ya se verificaron arriba)
def lim(filas):
    return filas[:LIM_DRY] if DRY else filas

TRAIN = {"M": lim(TRAIN_M), "S": lim(TRAIN_S)}
V_USO = lim(V)
_ids_v = {x["context_id"] for x in V_USO}
V_SONNET_USO = [x for x in V_SONNET if x["context_id"] in _ids_v]
GOLD_USO = lim(GOLD_V2_IDS)
print({"train_M": len(TRAIN["M"]), "train_S": len(TRAIN["S"]), "V": len(V_USO), "V_sonnet": len(V_SONNET_USO), "gold_v2": len(GOLD_USO)})

L2 quita 538 de 13331 (natural) y 561 de 13325 (sonnet)
  natural por clase: {'Application': 160, 'Background': 180, 'Basis': 23, 'Comparison': 87, 'Evidence': 37, 'Further Reading': 18, 'Gap': 13, 'Identification of the Originator': 9, 'Improvement / Modification': 11}
  sonnet  por clase: {'Application': 79, 'Background': 117, 'Basis': 27, 'Comparison': 148, 'Evidence': 104, 'Further Reading': 40, 'Gap': 29, 'Identification of the Originator': 4, 'Improvement / Modification': 13}
L3: {'citas_v_a_citantes_gold': 0, 'citas_v': 3379}
{'train_M': 12793, 'train_S': 12764, 'V': 3379, 'V_sonnet': 3379, 'gold_v2': 1533}


In [4]:
# Preregistro: se escribe UNA vez, antes de cualquier run, y nunca se sobrescribe.
PRE = EVIDENCIAS / "preregistro_b.json"
if not PRE.exists():
    escribir_json("preregistro_b.json", {
        "escrito_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "git": rm.estado_git(),
        "notebook_sha256": sd.sha_archivo(REPO / "notebooks/06_experimentos_104_entrada_b.ipynb"),
        "pistas": PISTAS, "hiperparametros": HP, "semillas": SEMILLAS_PLAN, "dry": DRY,
        "L4": {"umbral": UMBRAL_L4, "shingle_palabras": eb.K_SHINGLE, "descartar": L4_DESCARTAR,
               "accion": "citados con algun chunk con solape >= umbral se descartan del train/V (en A y B) y L4 se re-verifica en 0"},
        "primaria": "B - A, F1-macro sobre V_B con etiqueta propia; bootstrap por citante",
        "regla": "B ayuda en la pista X solo si las 3 parejas de semillas tienen el mismo signo y el IC95 excluye 0",
        "test_gold": "solo A vs B por pista y semilla sobre Gold v2 interseccion cobertura; sin elegir entre corridas"})
print("preregistro:", PRE, json.loads(PRE.read_text(encoding="utf-8"))["escrito_utc"])

preregistro: /home/carlos/microproyecto-local-citation/proyecto_de_grado/docs/evidencias_experimentos_104_b/preregistro_b.json 2026-11-14T16:22:33+00:00


## Fase 1 — Citados requeridos y descarga (PASO CON RED, solo de día)

Con B hay que tener el texto completo (ACL-OCL) de cada citado. Faltan ~1.830 de 3.363 en `cache_ocl`. Con `DESCARGAR = True` se bajan del
dataset público de HuggingFace (se envían solo ids de artículos ACL; 30–60 min). De noche **no** se usa la red: con `DESCARGAR = False`
se sigue con lo que haya en caché y la cobertura queda por debajo de lo esperado (se avisa).

In [5]:
CONJ = {"train_M": eb.casos_para_chunks(TRAIN["M"], CTX), "train_S": eb.casos_para_chunks(TRAIN["S"], CTX),
        "V": eb.casos_para_chunks(V_USO, CTX), "gold_v2": eb.casos_para_chunks([{"context_id": c} for c in GOLD_USO], CTX)}
REQUERIDOS = sorted({c["cited_id"] for casos in CONJ.values() for c in casos})
CACHE_OCL = ad.CACHE_OCL


def en_cache(cited: str) -> bool:
    return (CACHE_OCL / f"{cited}.json").is_file() or (CACHE_OCL / f"{cited}.404").exists()


FALTAN = [c for c in REQUERIDOS if not en_cache(c)]
print(f"citados requeridos: {len(REQUERIDOS)} | sin descargar: {len(FALTAN)}")
if FALTAN and DESCARGAR:
    lista = EVIDENCIAS / "citados_requeridos_b.txt"
    lista.write_text("".join(f"{c}\n" for c in FALTAN), encoding="utf-8")
    n_previos = len(list(EVIDENCIAS.glob("manifiesto_ocl_b_*.json")))
    rc = descarga_ocl.main(["--citados", str(lista), "--cache", str(CACHE_OCL),
                            "--manifiesto", str(EVIDENCIAS / f"manifiesto_ocl_b_{n_previos + 1}.json")])
    print("descarga_ocl rc =", rc)
    FALTAN = [c for c in REQUERIDOS if not en_cache(c)]
    print("siguen sin descargar:", len(FALTAN))
elif FALTAN:
    print(f"AVISO: DESCARGAR = False y faltan {len(FALTAN)} citados: la cobertura B sera menor a la esperada (~55 %).")

citados requeridos: 3658 | sin descargar: 1784
Descarga OCL | 1784 citados | {'descargado': 1766, '404': 10, 'error': 8} | /home/carlos/microproyecto-local-citation/proyecto_de_grado/docs/evidencias_experimentos_104_b/manifiesto_ocl_b_1.json
descarga_ocl rc = 1
siguen sin descargar: 8


## Fase 2 — Chunks y Top-3 (GPU)

Mismo recuperador que A1/A4 (SciBERT base sin ajustar, revisión de A1) y la misma heurística `perfil_seccion`: ninguno ve etiquetas (**L6**).
Los embeddings se reutilizan de la caché de A4. Reanudable: si `top3_<conjunto>.csv` ya existe con los mismos contextos, no se recalcula.
**Regresión L6**: en los contextos compartidos con los `top3_*.csv` de A4, los `chunk_id` y su orden deben coincidir.

In [6]:
CASOS_CHUNKS, EXCL_CHUNKS = servicio.construir_casos_split(CONJ, CTX, PAPERS, CACHE_OCL, ad.integrar, ad.fragmentar)
print({n: f"{len(c)}/{len(CONJ[n])}" for n, c in CASOS_CHUNKS.items()})

MODELO_RECUPERADOR = cli.EVIDENCIAS / "manifiesto_modelo_scibert.json"
CODIFICADOR, CACHE_EMB, HUELLAS, T_REC = None, CacheDisco(cli.CACHE_EMBEDDINGS), {}, 0.0
TOP3_FILAS = {}
for nombre, casos in CASOS_CHUNKS.items():
    ruta = EVIDENCIAS / f"top3_{nombre}.csv"
    esperados = {c["context_id"] for c in casos}
    if ruta.is_file() and {f["context_id"] for f in cli.leer_csv_top3(ruta)} == esperados:
        TOP3_FILAS[nombre] = cli.leer_csv_top3(ruta)   # reanudacion: la GPU ya hizo este conjunto
        continue
    if not casos:
        TOP3_FILAS[nombre] = []
        continue
    if CODIFICADOR is None:
        CODIFICADOR = cli_modelo.fabrica_recuperador(argparse.Namespace(
            device="auto", batch_size=16, max_length=512, manifiesto_modelo=MODELO_RECUPERADOR))
    t = time.time()
    res = servicio.recuperar_top3(casos, CODIFICADOR, CACHE_EMB, max_length=512,
                                  pooling=getattr(CODIFICADOR, "pooling", "mean"))
    T_REC += time.time() - t
    HUELLAS[nombre] = res["huella"]
    TOP3_FILAS[nombre] = cli._filas_top3(res["top3"])
    ruta.write_text(cli._csv_texto(cli.COL_TOP3, TOP3_FILAS[nombre]), encoding="utf-8", newline="")
    print(f"top3 {nombre}: {len(casos)} contextos, {res['articulos']} articulos, {time.time() - t:.0f}s")


# L6: regresion contra los top3_*.csv de A4 (mismos chunk_id y orden en los contextos compartidos)
def como_top3(filas):
    return {ctx: [{"posicion": r["posicion"], "chunk_id": r["chunk_id"], "similitud_coseno": float(r["puntaje"])}
                  for r in rs] for ctx, rs in servicio.agrupar_top3(filas).items()}


# eval_2015 y gold_v2 de A4 comparten contextos (ambos son test 2015): se unen sin duplicar y se exige que coincidan entre si
_A4 = {}
for n in ("train", "val", "eval_2015", "gold_v2"):
    for f in cli.leer_csv_top3(cli.EVIDENCIAS / f"top3_{n}.csv"):
        previa = _A4.setdefault((f["context_id"], f["posicion"]), f)
        assert (previa["chunk_id"], previa["puntaje"]) == (f["chunk_id"], f["puntaje"]), f"A4 inconsistente consigo mismo: {f['context_id']}"
A4_FILAS = list(_A4.values())
REGRESION = {n: servicio.regresion_top3(como_top3(f), A4_FILAS) for n, f in TOP3_FILAS.items() if f}
for n, r in REGRESION.items():
    print(f"L6 {n}: {r['coinciden']}/{r['contextos_comparados']} contextos comparados con A4, dif max similitud {r['max_dif_similitud']:.1e}")
    assert not r["discrepancias"], f"Regresion contra A4 fallida en {n}: {r['discrepancias'][:5]}"
if not DRY:
    assert REGRESION["gold_v2"]["contextos_comparados"] > 0, "La regresion contra A4 no comparo ningun contexto"

{'train_M': '6995/12793', 'train_S': '6913/12764', 'V': '1816/3379', 'gold_v2': '840/1533'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 41295.59it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

top3 train_M: 6995 contextos, 1550 articulos, 385s
top3 train_S: 6913 contextos, 1508 articulos, 153s
top3 V: 1816 contextos, 628 articulos, 63s
top3 gold_v2: 840 contextos, 488 articulos, 49s
L6 train_M: 1144/1144 contextos comparados con A4, dif max similitud 0.0e+00
L6 train_S: 1144/1144 contextos comparados con A4, dif max similitud 0.0e+00
L6 V: 246/246 contextos comparados con A4, dif max similitud 0.0e+00
L6 gold_v2: 840/840 contextos comparados con A4, dif max similitud 0.0e+00


## Fase 3 — Intersección A/B, cobertura, controles L4–L5 y manifiesto

`servicio.interseccion`: un contexto entra si tiene chunks, un Top-3 completo del artículo citado y título/abstract. Cobertura por conjunto y clase
(se espera ≈55 %). Luego **L4** (texto: ningún chunk que entra a un input comparte ≥50 % de shingles de 8 palabras con un contexto de V o del Gold) y **L5** (A y B usan exactamente los mismos ids).
L4 detecta lo que L2 no ve: el mismo artículo bajo otro ID (p. ej. `P97-1063` en el train y su gemelo `E97-1063` como citante de V) o texto reutilizado por los mismos autores.
Con `L4_DESCARTAR = True` los citados con algún chunk que solapa se **descartan del train/V en A y en B** (como L2, con conteo por clase) y L4 se vuelve a correr: lo que
entra a los inputs debe quedar en 0. Con `False` el notebook aborta en su lugar. Todo bloqueante queda en `fugas_b.json`.

In [7]:
TOP3 = {n: servicio.agrupar_top3(f) for n, f in TOP3_FILAS.items()}
INTER = servicio.interseccion(CONJ, CASOS_CHUNKS, EXCL_CHUNKS, TOP3, PAPERS)
ENTRADAS_B = eb.entradas_por_contexto(INTER["entradas"])
COBERTURA = INTER["resumen"]
for n, r in COBERTURA.items():
    print(f"{n:8s} candidatos={r['candidatos']:6d} interseccion={r['interseccion']:6d} cobertura={r['cobertura_pct']}% excluidos={r['excluidos_por_motivo']}")
print(pd.DataFrame({"candidatos": COBERTURA["train_M"]["por_clase_candidatos"],
                    "con B": COBERTURA["train_M"]["por_clase_interseccion"]})
      .assign(**{"cobertura %": lambda d: (100 * d["con B"] / d["candidatos"]).round(1)}).to_string())


def cubiertas(filas):
    return [x for x in filas if x["context_id"] in ENTRADAS_B]


TRAIN_B = {k: cubiertas(v) for k, v in TRAIN.items()}
V_B = {"M": cubiertas(V_USO), "S": cubiertas(V_SONNET_USO)}
GOLD_B = [c for c in GOLD_USO if c in ENTRADAS_B]
print({"train_M_B": len(TRAIN_B["M"]), "train_S_B": len(TRAIN_B["S"]), "V_B": len(V_B["M"]), "V_sonnet_B": len(V_B["S"]),
       "gold_v2_B": len(GOLD_B)})
assert all(TRAIN_B[k] and V_B[k] for k in PISTAS), "Sin cobertura B: revise la descarga (Fase 1)"

train_M  candidatos= 12793 interseccion=  6988 cobertura=54.62% excluidos={'ValueError': 22, 'autocita': 6, 'menos_de_3_chunks': 7, 'ocl_404': 33, 'parrafo_largo': 5726, 'sin_descargar': 11}
train_S  candidatos= 12764 interseccion=  6906 cobertura=54.11% excluidos={'ValueError': 21, 'autocita': 12, 'menos_de_3_chunks': 7, 'ocl_404': 34, 'parrafo_largo': 5776, 'sin_descargar': 8}
V        candidatos=  3379 interseccion=  1814 cobertura=53.68% excluidos={'ValueError': 6, 'autocita': 4, 'menos_de_3_chunks': 2, 'ocl_404': 3, 'parrafo_largo': 1548, 'sin_descargar': 2}
gold_v2  candidatos=  1533 interseccion=   839 cobertura=54.73% excluidos={'menos_de_3_chunks': 1, 'parrafo_largo': 693}
                                  candidatos  con B  cobertura %
Background                              5174   2853         55.1
Gap                                      435    254         58.4
Basis                                    577    301         52.2
Comparison                              1323    7

In [8]:
# L4: texto de los chunks que de verdad entran a los inputs vs contextos de V y del Gold
def chunks_usados(filas):
    return [{"context_id": x["context_id"], "cited_id": ENTRADAS_B[x["context_id"]]["cited_id"],
             "chunk_id": c["chunk_id"], "texto": c["texto"]}
            for x in filas for c in ENTRADAS_B[x["context_id"]]["top3"]]


CONTEXTOS_EVAL = {"V": {x["context_id"]: CTX[x["context_id"]]["masked_text"] for x in V},
                  "gold": {c: CTX[c]["masked_text"] for c in sorted(FUENTES.ids_gold) if c in CTX}}


def correr_l4():
    return eb.solapes_texto({"train_M": chunks_usados(TRAIN_B["M"]), "train_S": chunks_usados(TRAIN_B["S"]),
                             "V": chunks_usados(V_B["M"])}, CONTEXTOS_EVAL, umbral=UMBRAL_L4)


def mostrar_l4(reporte, titulo):
    for par, r in reporte["pares"].items():
        print(f"{titulo} {par}: {r['chunks_unicos']} chunks, max solape {r['max_solape']}, violaciones {r['violaciones']}, histograma {r['histograma']}")


t = time.time()
L4_INICIAL = correr_l4()
mostrar_l4(L4_INICIAL, "L4")
print(f"L4 en {time.time() - t:.0f}s")
L4_DESCARTES = {"citados": {}, "citas_por_clase": {}}
if L4_INICIAL["total_violaciones"] and L4_DESCARTAR:
    for conjunto, destinos in (("train_M", [(TRAIN_B, "M")]), ("train_S", [(TRAIN_B, "S")]), ("V", [(V_B, "M"), (V_B, "S")])):
        contaminados = eb.citados_con_solape(L4_INICIAL, conjunto)
        L4_DESCARTES["citados"][conjunto] = sorted(contaminados)
        for dic, k in destinos:
            dic[k], quitadas = eb.excluir_citas_a_citantes(dic[k], CTX, contaminados)
            L4_DESCARTES["citas_por_clase"][f"{conjunto}:{k}"] = quitadas
    print("L4 descarta:", {n: len(c) for n, c in L4_DESCARTES["citados"].items()}, "citados |", L4_DESCARTES["citas_por_clase"])
    L4 = correr_l4()   # tras descartar: lo que entra a los inputs debe quedar en 0
    mostrar_l4(L4, "L4 (tras descartar)")
else:
    L4 = L4_INICIAL
eb.exigir_sin_solapes(L4)
assert all(TRAIN_B[k] and V_B[k] for k in PISTAS), "L4 dejo una pista sin datos"
print({"train_M_B": len(TRAIN_B["M"]), "train_S_B": len(TRAIN_B["S"]), "V_B": len(V_B["M"]), "V_sonnet_B": len(V_B["S"])})

# L5: A y B usan exactamente los mismos ids (por pista, train y V_B)
IDS_SHA = {}
for k in PISTAS:
    for parte, filas in (("train", TRAIN_B[k]), ("val", V_B[k])):
        a, _ = ex.casos_a(filas, FUENTES, PAPERS)
        b, _ = ex.casos_b(filas, FUENTES, PAPERS, ENTRADAS_B)
        assert len(a) == len(filas), "A omitio citas sin titulo/abstract: A y B no usarian las mismas citas"
        IDS_SHA[f"{k}_{parte}"] = eb.verificar_mismos_ids(f"{k}_{parte}", a, b)
print("L5:", {n: s[:12] for n, s in IDS_SHA.items()})

USADOS = sorted({x["context_id"] for k in PISTAS for x in (*TRAIN_B[k], *V_B[k])})
TOP3_SHA = servicio.huella_entradas([ENTRADAS_B[c] for c in USADOS], "B")
SHA_DATOS_B = hashlib.sha256(json.dumps(IDS_SHA, sort_keys=True).encode("utf-8")).hexdigest()

L4 train_M~V: 9436 chunks, max solape 0.9565, violaciones 4, histograma {'(0,0.1]': 59, '(0.1,0.2]': 12, '(0.2,0.3]': 3, '(0.3,0.4]': 2, '0': 9356, '>0.5': 4}
L4 train_M~gold: 9436 chunks, max solape 0.2708, violaciones 0, histograma {'(0,0.1]': 47, '(0.1,0.2]': 1, '(0.2,0.3]': 2, '0': 9386}
L4 train_S~V: 9239 chunks, max solape 0.9565, violaciones 4, histograma {'(0,0.1]': 57, '(0.1,0.2]': 11, '(0.2,0.3]': 2, '(0.3,0.4]': 2, '0': 9163, '>0.5': 4}
L4 train_S~gold: 9239 chunks, max solape 0.7097, violaciones 1, histograma {'(0,0.1]': 53, '(0.1,0.2]': 1, '(0.2,0.3]': 2, '0': 9182, '>0.5': 1}
L4 V~gold: 3362 chunks, max solape 0.234, violaciones 0, histograma {'(0,0.1]': 25, '(0.1,0.2]': 3, '(0.2,0.3]': 1, '0': 3333}
L4 en 4s
L4 descarta: {'train_M': 1, 'train_S': 4, 'V': 0} citados | {'train_M:M': {'Application': 4, 'Background': 3, 'Comparison': 1, 'Evidence': 1}, 'train_S:S': {'Application': 5, 'Background': 10, 'Basis': 4, 'Comparison': 4, 'Evidence': 6, 'Gap': 1, 'Identification of t

In [9]:
# L8 (solo reporte): citados compartidos entre train y V/Gold; Gold que cita a citantes de V/train
CITADOS_TRAIN = {k: {CTX[x["context_id"]]["refid"] for x in TRAIN_B[k]} for k in PISTAS}
citantes_train = {CTX[x["context_id"]]["citing_id"] for k in PISTAS for x in TRAIN_B[k]}
CITANTES_V_Y_TRAIN = {x["citing_id"] for x in V} | citantes_train
L8 = {"v_con_citado_visto_en_train": {k: sum(CTX[x["context_id"]]["refid"] in CITADOS_TRAIN[k] for x in V_B[k]) for k in PISTAS},
      "v_n": {k: len(V_B[k]) for k in PISTAS},
      "gold_v2_cita_citante_de_v_o_train": sum(CTX[c]["refid"] in CITANTES_V_Y_TRAIN for c in GOLD_USO)}
print("L8:", L8)

BLOQUEANTES = {"L1_fugas_contexto": L1["total_fugas"], "L3_citas_v_a_citantes_gold": 0,   # verificar_citados_v ya lanzo si > 0
               "L4_violaciones_texto": L4["total_violaciones"],
               "L5_ids_ab_distintos": 0,                                                   # verificar_mismos_ids ya lanzo si difieren
               "L6_discrepancias_top3": sum(len(r["discrepancias"]) for r in REGRESION.values())}
FUGAS = {"dry": DRY, "bloqueantes": BLOQUEANTES, "L1": L1, "L2_citas_quitadas_por_clase": {"natural": QUITADAS_M, "sonnet": QUITADAS_S},
         "L3": L3, "L4": L4, "L4_inicial": L4_INICIAL, "L4_descartes": L4_DESCARTES, "L5_sha_ids": IDS_SHA, "L6": REGRESION,
         "L7": "El Test Gold anotado no se carga; del Gold v2 solo ids; las predicciones se escriben sin etiquetas",
         "L8_solo_reporte": L8}
escribir_json("fugas_b.json", FUGAS)
assert sum(BLOQUEANTES.values()) == 0, BLOQUEANTES
print("fugas_b.json: todo bloqueante en 0")

escribir_json("manifiesto_datos_b.json", {
    "dry": DRY, "git": rm.estado_git(), "top3_sha": TOP3_SHA, "sha256_ids_pistas": IDS_SHA,
    "n": {"train_M_B": len(TRAIN_B["M"]), "train_S_B": len(TRAIN_B["S"]), "V_B": len(V_B["M"]), "V_sonnet_B": len(V_B["S"]),
          "gold_v2": len(GOLD_USO), "gold_v2_B": len(GOLD_B)},
    "sha256_ids": {"train_M_B": dominio.sha_ids(x["context_id"] for x in TRAIN_B["M"]),
                   "train_S_B": dominio.sha_ids(x["context_id"] for x in TRAIN_B["S"]),
                   "V_B": dominio.sha_ids(x["context_id"] for x in V_B["M"]),
                   "V_sonnet_B": dominio.sha_ids(x["context_id"] for x in V_B["S"]),
                   "gold_v2_B": dominio.sha_ids(GOLD_B)},
    "datos_104_sha256_ids": SHA_ACTUAL, "cobertura": COBERTURA,
    "sha256_top3_csv": {n: sd.sha_archivo(EVIDENCIAS / f"top3_{n}.csv") for n in TOP3_FILAS if (EVIDENCIAS / f"top3_{n}.csv").is_file()},
    "hiperparametros": HP, "semillas": SEMILLAS})
excl = cli._csv_texto(cli.COL_EXCLUIDOS, INTER["excluidos"])
(EVIDENCIAS / "excluidos_b.csv").write_text(excl, encoding="utf-8", newline="")
escribir_json("manifiesto_recuperacion_b.json", {
    "encoder": CODIFICADOR.info() if CODIFICADOR is not None else "reanudado: top3 ya calculado",
    "sha256_manifiesto_modelo": cli._sha_manifiesto(MODELO_RECUPERADOR), "huellas": HUELLAS, "segundos": round(T_REC, 1),
    "cobertura_chunks": {n: [len(c), len(CONJ[n])] for n, c in CASOS_CHUNKS.items()}, "regresion_a4": REGRESION})

L8: {'v_con_citado_visto_en_train': {'M': 1475, 'S': 1475}, 'v_n': {'M': 1814, 'S': 1814}, 'gold_v2_cita_citante_de_v_o_train': 737}
fugas_b.json: todo bloqueante en 0


PosixPath('/home/carlos/microproyecto-local-citation/proyecto_de_grado/docs/evidencias_experimentos_104_b/manifiesto_recuperacion_b.json')

## Fase 4 — Núcleo: 2 pistas × {A, B} × 3 semillas = 12 corridas de SciBERT

Mismas citas, etiquetas, hiperparámetros y semillas; solo cambia el input. Early stopping sobre V_B (de la pista) en A **y** en B. Se alterna
pista/entrada dentro de cada semilla: si se corta por tiempo queda una cobertura balanceada. Reanudable (`config_sha`).

In [10]:
def config_pista(k: str) -> dict:
    p = PISTAS[k]
    cfg = {"pista": p["pista"], "variante": p["variante"], "v_sha": IDS_SHA[f"{k}_val"],
           "ids_train_sha": IDS_SHA[f"{k}_train"], "cobertura": "interseccion_B"}
    return {**cfg, "v_etiquetas": "sonnet"} if k == "S" else cfg


def run(k: str, entrada: str, semilla: int, encoder: str = "scibert"):
    nombre = f"{encoder}-{entrada}-{k}-s{semilla}"
    run_id = correr(nombre, ex.entrenar_y_registrar, config_pista(k), TRAIN_B[k], V_B[k], FUENTES, PAPERS, CORRIDAS,
                    encoder=encoder, semilla=semilla, modo=PISTAS[k]["modo"], hp=HP_OBJ, dry=DRY,
                    manifiesto_datos_sha=SHA_DATOS_B, entrada=entrada, entradas_b=ENTRADAS_B,
                    top3_sha=TOP3_SHA if entrada != "A" else "")
    LISTA_RUNS.append({"nombre": nombre, "encoder": encoder, "entrada": entrada, "pista": k, "semilla": semilla, "run_id": run_id})
    return run_id


for s in SEMILLAS:
    for k in PISTAS:
        for entrada in ("A", "B"):
            run(k, entrada, s)
print("fallas:", FALLAS or "ninguna", f"| {(time.time() - T0) / 3600:.2f} h")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26652.19it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    393s] scibert-A-M-s42 -> 70c16daaeb16473b883e3dda95e6e5b8


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 38377.24it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    628s] scibert-B-M-s42 -> 8abf2dbc7201495f908108d0c534b8a2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 54104.26it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    381s] scibert-A-S-s42 -> 853d463b96e84ea98c9cbafcfff068e3


... (14 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 31097.86it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    375s] scibert-A-S-s2026 -> d17693c9b68145c69790b9c277df8658


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25985.88it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    609s] scibert-B-S-s2026 -> dbd65fba6bc54f93a4f6122b79660c5f
fallas: ninguna | 1.95 h


## Fase 5 — Referencia: los ganadores de #104 (entrada A, entrenados con todo el train) sobre V_B

Sin reentrenar y sin input B: sus `predicciones_val.jsonl` restringidas a V_B. La diferencia con las corridas A de este notebook es el costo de
entrenar con menos citas (solo las de cobertura B).

In [11]:
ETIQ_V = {k: {x["context_id"]: dominio.normalizar_etiqueta(x["etiqueta_entrenamiento"]) for x in V_B[k]} for k in PISTAS}


def f1_en(preds: dict, etiquetas: dict) -> tuple[float, int]:
    ids = sorted(set(preds) & set(etiquetas))
    return dominio.metricas([etiquetas[c] for c in ids], [preds[c] for c in ids])["f1_macro"], len(ids)


def preds_val(nombre: str, base: Path = CORRIDAS):
    carpeta = base / nombre
    return ex.leer_predicciones(carpeta) if (carpeta / "predicciones_val.jsonl").is_file() else None


REFERENCIA = {}
for k, stem in REFERENCIA_104.items():
    for s in SEMILLAS_PLAN:
        p = preds_val(f"{stem}-s{s}", DIR_104)
        if p:
            REFERENCIA[f"{k}-s{s}"] = dict(zip(("f1_macro", "n"), f1_en(p, ETIQ_V[k])))
print(pd.DataFrame(REFERENCIA).T.round(4).to_string() if REFERENCIA else "sin predicciones de los ganadores de #104")

         f1_macro       n
M-s42      0.7179  1814.0
M-s1234    0.7113  1814.0
M-s2026    0.7014  1814.0
S-s42      0.6552  1814.0
S-s1234    0.6595  1814.0
S-s2026    0.6249  1814.0


## Fase 6 — Opcionales (según `HORAS_MAX`): `B_barajada` y SPECTER

`B_barajada` permuta los perfiles de sección entre los chunks del conjunto (conserva la marginal): mide si la sección aporta señal. SPECTER A/B s42 completa el
2×2 que pide #103 (los hiperparámetros no se ajustan por encoder).

In [12]:
if CORRER_BARAJADA:
    for s in SEMILLAS:
        for k in PISTAS:
            run(k, "B_barajada", s)
if CORRER_SPECTER:
    for k in PISTAS:
        for entrada in ("A", "B"):
            run(k, entrada, 42, encoder="specter")
print("fallas:", FALLAS or "ninguna", f"| {(time.time() - T0) / 3600:.2f} h")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 38717.25it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    623s] scibert-B_barajada-M-s42 -> aa4b3b8867d146ca91730a36bd991074


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24286.15it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    622s] scibert-B_barajada-S-s42 -> 29a21aa4379148c9acbdca801074c10b


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23644.27it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    652s] scibert-B_barajada-M-s1234 -> c648491392e54c94909f9fa67a165503


... (10 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26965.61it/s]


[ok    378s] specter-A-S-s42 -> a8037d0f7faf4c3aa6b5e31dbdc8c74f


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 36486.56it/s]


[ok    602s] specter-B-S-s42 -> 99a98bc39dda4328b0fa6cb823561a14
fallas: ninguna | 3.55 h


## Fase 6b — Smoke test de inferencia

Un checkpoint B por pista (y uno `B_barajada` si existe), semilla 42: se recarga verificando su sha256 y debe reproducir las predicciones de `val` guardadas al entrenar.

In [13]:
SMOKE = []
for k in PISTAS:
    for entrada in ("B", "B_barajada"):
        carpeta = CORRIDAS / f"scibert-{entrada}-{k}-s42"
        if not (carpeta / "model.safetensors").is_file():
            continue
        try:
            r = ex.smoke_inferencia(carpeta, "scibert", V_B[k], FUENTES, PAPERS, entrada=entrada, entradas_b=ENTRADAS_B)
        except Exception:
            FALLAS.append(f"smoke-{k}-{entrada}")
            r = {"ok": False, "error": traceback.format_exc()}
        SMOKE.append({"pista": k, "entrada": entrada, **r})
        print({c: v for c, v in SMOKE[-1].items() if c != "error"})
        liberar_gpu()
escribir_json("smoke_inferencia_b.json", SMOKE)
assert SMOKE and all(r["ok"] for r in SMOKE) or not hay_tiempo(), "El smoke test no reproduce las predicciones guardadas"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 41608.50it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'pista': 'M', 'entrada': 'B', 'run': 'scibert-B-M-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 5.707070750020193, 'checkpoint_sha256': 'e1f0e89f1de96e13647100fa6f820745e48c363d27693dea1b7f71b25cd85cfe'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25720.83it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'pista': 'M', 'entrada': 'B_barajada', 'run': 'scibert-B_barajada-M-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 5.724779656247847, 'checkpoint_sha256': 'f57f9c4f9ec4ecc62ec083953fdc679ad4d60e2a02a0149d6d3cdceab348c2c0'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17571.56it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'pista': 'S', 'entrada': 'B', 'run': 'scibert-B-S-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 5.736339718737327, 'checkpoint_sha256': '8cc995d2fabb67bea0dbc60eb89e44559ccdc72ef7f7db8042a405f723633360'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 27718.73it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'pista': 'S', 'entrada': 'B_barajada', 'run': 'scibert-B_barajada-S-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 5.726745343736184, 'checkpoint_sha256': '59f3ce0588b078ecf4e1abd2bb7fa251b8f6d82284479f21dc025fca3f6a8901'}


## Fase 7 — Predicciones ciegas sobre el Gold v2 (sin leer etiquetas)

Para cada run A/B: B sobre Gold v2 ∩ cobertura; A sobre Gold v2 ∩ cobertura (conjunto pareado) y además sobre **todo** el Gold v2. Se guardan bajo `artifacts` con su sha256
en `manifiesto_gold_b.json`. Cada fila es `{context_id, pred, probs}`: el escritor rechaza cualquier otro campo (L7).

In [14]:
GOLD_A_TODO = ex.casos_gold(GOLD_USO, FUENTES, PAPERS, "A")
GOLD_A_COB = ex.casos_gold(GOLD_B, FUENTES, PAPERS, "A")
GOLD_B_COB = ex.casos_gold(GOLD_B, FUENTES, PAPERS, "B", ENTRADAS_B)
assert [c["context_id"] for c in GOLD_A_COB] == [c["context_id"] for c in GOLD_B_COB]   # conjunto pareado
PEDIDOS = {"A": {"cob": GOLD_A_COB, "todo": GOLD_A_TODO}, "B": {"cob": GOLD_B_COB}}

MANIFIESTO_GOLD = {}
for r in LISTA_RUNS:
    carpeta = CORRIDAS / r["nombre"]
    if r["entrada"] not in PEDIDOS or not (carpeta / "model.safetensors").is_file():
        continue
    try:
        archivos = {}
        for etiqueta, casos in PEDIDOS[r["entrada"]].items():
            ruta = GOLD_DIR / r["nombre"] / f"pred_gold_v2_{etiqueta}.jsonl"
            if not (ruta.is_file() and len(ruta.read_text(encoding="utf-8").splitlines()) == len(casos)):
                ex.escribir_predicciones_ciegas(ruta, ex.predecir_gold(carpeta, r["encoder"], casos, r["entrada"]))
            filas = [json.loads(l) for l in ruta.read_text(encoding="utf-8").splitlines()]
            assert all(set(f) == set(ex.CAMPOS_PREDICCION_CIEGA) for f in filas), "una prediccion lleva campos de mas"
            archivos[etiqueta] = {"n": len(filas), "sha256": sd.sha_archivo(ruta), "ruta": str(ruta.relative_to(REPO))}
        man = json.loads((carpeta / "manifiesto_entrenamiento.json").read_text(encoding="utf-8"))
        MANIFIESTO_GOLD[r["nombre"]] = {"encoder": r["encoder"], "entrada": r["entrada"], "pista": r["pista"], "semilla": r["semilla"],
                                        "checkpoint_sha256": man["checkpoint_sha256"], "archivos": archivos}
        print(f"[gold] {r['nombre']}: {({e: a['n'] for e, a in archivos.items()})}")
    except Exception:
        FALLAS.append(f"gold-{r['nombre']}")
        print(f"[FALLO] gold {r['nombre']}\n{traceback.format_exc()}")
    finally:
        liberar_gpu()
escribir_json("manifiesto_gold_b.json", {"dry": DRY, "gold_v2": len(GOLD_USO), "gold_v2_cobertura": len(GOLD_B),
                                         "ids_gold_v2_cobertura_sha256": dominio.sha_ids(GOLD_B), "runs": MANIFIESTO_GOLD})

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 36334.08it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[gold] scibert-A-M-s42: {'cob': 839, 'todo': 1533}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 46928.29it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[gold] scibert-B-M-s42: {'cob': 839}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24376.94it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[gold] scibert-A-S-s42: {'cob': 839, 'todo': 1533}


... (23 outputs left out)

[gold] specter-A-S-s42: {'cob': 839, 'todo': 1533}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35352.24it/s]


[gold] specter-B-S-s42: {'cob': 839}


PosixPath('/home/carlos/microproyecto-local-citation/proyecto_de_grado/docs/evidencias_experimentos_104_b/manifiesto_gold_b.json')

## Fase 8 — Análisis preregistrado y exportación

Primaria por pista: B − A de F1-macro sobre V_B (etiqueta propia), por semilla y promedio de semillas, con bootstrap por clústeres (artículo citante).
`afirmable` solo si **las 3 parejas de semillas tienen el mismo signo y el IC95 excluye 0**. Todo lo demás es secundario.

In [15]:
PIDS = {}   # {(pista, entrada, semilla): {context_id: pred}} de las predicciones de val de cada run


def preds_de(k, entrada, s, encoder="scibert"):
    clave = (k, entrada, s, encoder)
    if clave not in PIDS:
        PIDS[clave] = preds_val(f"{encoder}-{entrada}-{k}-s{s}")
    return PIDS[clave]


def comparar(k, base, otra, encoder="scibert", conjunto=None):
    # `otra` - `base` en F1-macro (promedio de semillas) sobre los ids comunes de V_B, bootstrap por citante
    sem = [s for s in SEMILLAS if preds_de(k, base, s, encoder) and preds_de(k, otra, s, encoder)]
    if not sem:
        return None
    ids = sorted(set.intersection(*[set(preds_de(k, e, s, encoder)) for s in sem for e in (base, otra)])
                 & (set(conjunto) if conjunto is not None else set(ETIQ_V[k])))
    if not ids:
        return None
    r = ex.bootstrap_diferencia_cluster(
        [[preds_de(k, base, s, encoder)[c] for c in ids] for s in sem],
        [[preds_de(k, otra, s, encoder)[c] for c in ids] for s in sem],
        [ETIQ_V[k][c] for c in ids], [CTX[c]["citing_id"] for c in ids], n=N_BOOTSTRAP, semilla=2026)
    r["semillas_usadas"] = sem
    r["ic_excluye_0"] = bool(r["ic95"][0] > 0 or r["ic95"][1] < 0)
    r["afirmable"] = bool(len(sem) == len(SEMILLAS_PLAN) and r["mismo_signo_semillas"] and r["ic_excluye_0"])
    return r


BOOT = []
for k in PISTAS:
    for base, otra, nombre in (("A", "B", "B - A"), ("B", "B_barajada", "B_barajada - B")):
        if otra == "B_barajada" and not CORRER_BARAJADA:
            continue
        try:
            r = comparar(k, base, otra)
        except Exception:
            FALLAS.append(f"bootstrap-{k}-{nombre}")
            print(f"[FALLO] bootstrap {k} {nombre}\n{traceback.format_exc()}")
            continue
        if r is None:
            print(f"{k} {nombre}: sin corridas completas")
            continue
        BOOT.append({"pista": k, "comparacion": nombre, "primaria": nombre == "B - A", **r})
        print(f"{k} {nombre:15s} dif={r['diferencia']:+.4f} IC95=[{r['ic95'][0]:+.4f}, {r['ic95'][1]:+.4f}] "
              f"por semilla={[round(d, 4) for d in r['diferencia_por_semilla']]} n={r['casos']} citantes={r['grupos']} "
              f"-> afirmable: {r['afirmable']}")
escribir_json("bootstrap_b.json", BOOT)

M B - A           dif=-0.0211 IC95=[-0.0344, -0.0090] por semilla=[-0.0231, -0.0094, -0.0309] n=1814 citantes=362 -> afirmable: True
M B_barajada - B  dif=+0.0071 IC95=[-0.0030, +0.0176] por semilla=[-0.0071, 0.0166, 0.0119] n=1814 citantes=362 -> afirmable: False
S B - A           dif=-0.0116 IC95=[-0.0303, +0.0067] por semilla=[-0.0059, -0.0225, -0.0065] n=1814 citantes=362 -> afirmable: False
S B_barajada - B  dif=+0.0074 IC95=[-0.0047, +0.0205] por semilla=[0.0184, 0.0032, 0.0007] n=1814 citantes=362 -> afirmable: False


PosixPath('/home/carlos/microproyecto-local-citation/proyecto_de_grado/docs/evidencias_experimentos_104_b/bootstrap_b.json')

In [16]:
# Secundarias: F1-macro por semilla y en otros conjuntos (metricas que MLflow registro en cada run), media +- std entre semillas
def tabla_runs() -> pd.DataFrame:
    exp = mlflow.get_experiment_by_name(rm.EXPERIMENTO_B + SUFIJO)
    filas = []
    for r in mlflow.search_runs([exp.experiment_id], output_format="list"):
        t, m, p = r.data.tags, r.data.metrics, r.data.params
        if r.info.status != "FINISHED" or "invalido" in t or "val_f1_macro" not in m:
            continue
        nombre = t.get("mlflow.runName", "").rsplit("-", 2)[0]
        filas.append({"run": nombre, "pista": p.get("pista"), "entrada": p.get("entrada"), "encoder": p.get("encoder"),
                      "semilla": int(p.get("semilla", -1)),
                      **{k: m.get(k) for k in ("val_f1_macro", "val_mixta_f1_macro", "val_sonnet_f1_macro", "dev_haiku_f1_macro",
                                               "ref_gpt_f1_macro", "val_n", "segundos_entrenamiento", "gpu_mem_pico_gb")}})
    return pd.DataFrame(filas)


TABLA = tabla_runs()
COLS = ["val_f1_macro", "val_mixta_f1_macro", "val_sonnet_f1_macro", "dev_haiku_f1_macro", "ref_gpt_f1_macro"]
if len(TABLA):
    RESUMEN = TABLA.groupby(["pista", "encoder", "entrada"])[COLS + ["segundos_entrenamiento"]].agg(["mean", "std"]).round(4)
    print(RESUMEN.to_string())
    TABLA.sort_values(["pista", "encoder", "entrada", "semilla"]).to_csv(EVIDENCIAS / "resumen_b.csv", index=False)

                          val_f1_macro         val_mixta_f1_macro         val_sonnet_f1_macro         dev_haiku_f1_macro         ref_gpt_f1_macro         segundos_entrenamiento         
                                  mean     std               mean     std                mean     std               mean     std             mean     std                   mean      std
pista  encoder entrada                                                                                                                                                                   
mixta  scibert A                0.6835  0.0120             0.6835  0.0120              0.4822  0.0121             0.5648  0.0286           0.4253  0.0173               358.7667   7.3201
               B                0.6623  0.0067             0.6623  0.0067              0.4787  0.0037             0.5608  0.0036           0.4467  0.0103               585.3333   6.6214
               B_barajada       0.6695  0.0073             0.6695  0.0

In [17]:
# F1 por clase (media de semillas) y estratos citado visto / no visto en train, B - A
def metricas_semillas(k, entrada, ids):
    sem = [s for s in SEMILLAS if preds_de(k, entrada, s)]
    return [dominio.metricas([ETIQ_V[k][c] for c in ids], [preds_de(k, entrada, s)[c] for c in ids]) for s in sem]


ANALISIS = {"por_clase": {}, "estratos": {}}
for k in PISTAS:
    comunes = sorted(set.intersection(*[set(p) for s in SEMILLAS for e in ("A", "B") if (p := preds_de(k, e, s))])
                     & set(ETIQ_V[k])) if all(preds_de(k, e, s) for s in SEMILLAS for e in ("A", "B")) else []
    if not comunes:
        continue
    ma, mb = metricas_semillas(k, "A", comunes), metricas_semillas(k, "B", comunes)
    fa, fb = dominio.agregar_semillas(ma)["f1_por_clase"], dominio.agregar_semillas(mb)["f1_por_clase"]
    ANALISIS["por_clase"][k] = [{"clase": a["label"], "soporte": ma[0]["per_class"][i]["support"], "A": a["media"], "B": b["media"],
                                  "B-A": b["media"] - a["media"]} for i, (a, b) in enumerate(zip(fa, fb))]
    print(f"=== {k}: F1 por clase (media de semillas)")
    print(pd.DataFrame(ANALISIS["por_clase"][k]).round(3).to_string(index=False))
    for estrato, mascara in (("citado_visto_en_train", lambda c: CTX[c]["refid"] in CITADOS_TRAIN[k]),
                             ("citado_no_visto", lambda c: CTX[c]["refid"] not in CITADOS_TRAIN[k])):
        ids = [c for c in comunes if mascara(c)]
        if len(ids) < 2:
            continue
        fa_e = statistics.mean(m["f1_macro"] for m in metricas_semillas(k, "A", ids))
        fb_e = statistics.mean(m["f1_macro"] for m in metricas_semillas(k, "B", ids))
        ANALISIS["estratos"][f"{k}:{estrato}"] = {"n": len(ids), "A": fa_e, "B": fb_e, "B-A": fb_e - fa_e}
        print(f"  {estrato:22s} n={len(ids):5d} A={fa_e:.4f} B={fb_e:.4f} B-A={fb_e - fa_e:+.4f}")

=== M: F1 por clase (media de semillas)
                           clase  soporte     A     B    B-A
                      Background      721 0.882 0.876 -0.007
                             Gap       79 0.590 0.589 -0.001
                           Basis       68 0.515 0.475 -0.040
                      Comparison      220 0.800 0.784 -0.016
                     Application      475 0.907 0.902 -0.004
      Improvement / Modification       51 0.417 0.392 -0.025
                        Evidence      143 0.715 0.716  0.001
Identification of the Originator       30 0.598 0.526 -0.072
                 Further Reading       27 0.727 0.701 -0.026
  citado_visto_en_train  n= 1475 A=0.6874 B=0.6628 B-A=-0.0247
  citado_no_visto        n=  339 A=0.6743 B=0.6601 B-A=-0.0141
=== S: F1 por clase (media de semillas)
                           clase  soporte     A     B    B-A
                      Background      831 0.857 0.855 -0.002
                             Gap       52 0.480 0.503  0.023
 

In [18]:
# Truncamiento (SciBERT): fraccion de casos de V_B con recorte por segmento, A vs B
TOK = ct.TokenizadorHF(ct.resolver_encoder("scibert", MODELO_RECUPERADOR))
ANALISIS["truncamiento"] = {}
for k in PISTAS:
    for entrada in ("A", "B"):
        casos, _ = ex.casos_entrada(entrada, V_B[k], FUENTES, PAPERS, ENTRADAS_B)
        ej = ct.preparar_ejemplos([{c: v for c, v in x.items() if c != "_fila"} for x in casos], entrada, TOK, HP_OBJ.max_len)
        res = ct.resumen_truncamiento(ej)
        ANALISIS["truncamiento"][f"{k}:{entrada}"] = res
        print(f"{k} {entrada}: longitud media {res['longitud_media']:.0f} | " +
              ", ".join(f"{s}: {v['frac_truncado']:.2f}" for s, v in res["segmentos"].items()))
ANALISIS["referencia_104_sobre_V_B"] = REFERENCIA
ANALISIS["cobertura"] = COBERTURA
escribir_json("analisis_b.json", ANALISIS)

DF = rm.exportar_manifiesto(EVIDENCIAS, rm.EXPERIMENTO_B + SUFIJO)
print(len(DF), "runs exportados a", EVIDENCIAS)
print("fallas:", FALLAS or "ninguna", f"| {(time.time() - T0) / 3600:.2f} h")

M A: longitud media 227 | abstract: 0.01, contexto: 0.00, titulo: 0.00
M B: longitud media 481 | chunk_1: 0.37, chunk_2: 0.40, chunk_3: 0.42, contexto: 0.00
S A: longitud media 227 | abstract: 0.01, contexto: 0.00, titulo: 0.00
S B: longitud media 481 | chunk_1: 0.37, chunk_2: 0.40, chunk_3: 0.42, contexto: 0.00
22 runs exportados a /home/carlos/microproyecto-local-citation/proyecto_de_grado/docs/evidencias_experimentos_104_b
fallas: ninguna | 3.60 h
